In [1]:
from __future__ import annotations
import torch
from torch import nn
from torch.optim import Optimizer
from transformers import PreTrainedModel, PreTrainedTokenizer
from typing import List, Optional, Union, Tuple, Any
from tqdm.auto import tqdm


class EmbedPoisoner(nn.Module):
    """
    Wraps a pretrained decoder-only LM and injects trainable adversarial embeddings
    in place of a special token. Only adversarial embeddings are trainable; the LM is frozen.

    Args:
        model (PreTrainedModel): The pretrained language model to attack.
        tokenizer (PreTrainedTokenizer): Tokenizer corresponding to the model.
        adv_token (str): Special token to replace with adversarial embeddings.
        num_tokens (int): Number of embeddings to inject per placeholder.
        universal (bool): If True, uses a single shared embedding block for all inputs.
        norm (Union[str,float]): Norm type ('inf' or p>=1) for projection.
        epsilon (float): Radius for the projection ball.
    """

    def __init__(
        self,
        model: PreTrainedModel,
        tokenizer: PreTrainedTokenizer,
        adv_token: str,
        num_tokens: int,
        universal: bool = False,
        norm: Union[str, float] = "inf",
        epsilon: float = 1.0,
    ) -> None:
        super().__init__()
        self.model: PreTrainedModel = model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

        self.tokenizer: PreTrainedTokenizer = tokenizer
        if adv_token not in tokenizer.get_vocab():
            tokenizer.add_special_tokens({"additional_special_tokens": [adv_token]})
            self.model.resize_token_embeddings(len(tokenizer))
        tokens = tokenizer.tokenize(adv_token)
        assert tokens == [adv_token], f"`adv_token` must map to a single token, got {tokens}"
        self.adv_token_id: int = tokenizer.convert_tokens_to_ids(adv_token)

        self.num_tokens: int = num_tokens
        self.universal: bool = universal
        self.norm: Union[str, float] = norm
        self.epsilon: float = epsilon
        self.adversarial_embeddings: Optional[nn.Parameter] = None

    def init_embeddings(self, batch_size: int) -> None:
        """
        Initialize adversarial embeddings.

        Args:
            batch_size (int): Number of samples in the batch.
        """
        embed_dim = self.model.get_input_embeddings().weight.size(1)
        device = next(self.model.parameters()).device
        if self.universal:
            data = torch.zeros(self.num_tokens, embed_dim, device=device)
        else:
            data = torch.zeros(batch_size, self.num_tokens, embed_dim, device=device)
        self.adversarial_embeddings = nn.Parameter(data)

    def parameters(self, recurse: bool = True):
        """
        Yield only the adversarial embeddings as trainable parameters.

        Returns:
            Iterator[nn.Parameter]: The adversarial embeddings parameter.
        """
        if self.adversarial_embeddings is not None:
            yield self.adversarial_embeddings

    def project(self) -> None:
        """
        Project the adversarial embeddings into the L-p ball of radius epsilon.
        """
        assert self.adversarial_embeddings is not None
        if self.norm == "inf":
            with torch.no_grad():
                self.adversarial_embeddings.clamp_(-self.epsilon, self.epsilon)
        else:
            p = float(self.norm)
            with torch.no_grad():
                self.adversarial_embeddings.renorm_(p, dim=-1, maxnorm=self.epsilon)

    def _prepare_for_training(self, user_texts: List[str], target_texts: List[str]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, int]:
        """
        Prepare embeddings, attention mask, and labels for training.

        Args:
            user_texts (List[str]): Input prompts.
            target_texts (List[str]): Desired model responses.

        Returns:
            inputs_embeds (torch.Tensor): Embeddings with adversarial block injected.
            attention_mask (torch.Tensor): Corresponding attention mask.
            labels (torch.Tensor): Token IDs for target_texts, masked outside response span.
            prefix_length (int): Length of each sequence after insertion (T_new).
        """
        assert self.adversarial_embeddings is not None
        batch_size = len(user_texts)
        # Build chat template with both user and assistant
        convos = [
            [
                {"role": "user", "content": user + " " + self.tokenizer.convert_ids_to_tokens(self.adv_token_id)},
                {"role": "assistant", "content": tgt},
            ]
            for user, tgt in zip(user_texts, target_texts)
        ]
        toks = self.tokenizer.apply_chat_template(
            convos,
            add_generation_prompt=False,
            continue_final_message=True,
            padding=True,
            padding_side="right",
            return_tensors="pt",
            return_dict=True,
        ).to(self.adversarial_embeddings.device)
        input_ids = toks.input_ids
        orig_mask = toks.attention_mask
        orig_emb = self.model.get_input_embeddings()(input_ids)
        B, T = input_ids.size()

        # locate placeholder positions
        positions = [(row == self.adv_token_id).nonzero(as_tuple=True)[0].item() for row in input_ids]
        N = self.num_tokens
        D = orig_emb.size(-1)
        T_new = T - 1 + N

        inputs_embeds = torch.zeros(B, T_new, D, device=orig_emb.device)
        attention_mask = torch.zeros(B, T_new, device=orig_emb.device)
        labels = torch.full((B, T_new), -100, device=orig_emb.device, dtype=torch.long)

        # tokenize target texts to get IDs and lengths
        tgt_enc = self.tokenizer(target_texts, add_special_tokens=False, padding=True, return_attention_mask=True, return_tensors="pt").to(
            orig_emb.device
        )
        tgt_ids = tgt_enc.input_ids
        tgt_lens = tgt_enc.attention_mask.sum(dim=1)

        for i in range(B):
            pos = positions[i]
            # prefix
            inputs_embeds[i, :pos] = orig_emb[i, :pos]
            attention_mask[i, :pos] = orig_mask[i, :pos]
            # adversarial block
            block = self.adversarial_embeddings if self.universal else self.adversarial_embeddings[i]
            inputs_embeds[i, pos : pos + N] = block
            attention_mask[i, pos : pos + N] = 1
            # suffix
            inputs_embeds[i, pos + N :] = orig_emb[i, pos + 1 :]
            attention_mask[i, pos + N :] = orig_mask[i, pos + 1 :]
            # labels
            length = tgt_lens[i].item()
            start = int(orig_mask[i].sum().item() - length - 1 + N)
            labels[i, start : start + length] = tgt_ids[i, :length]

        return inputs_embeds, attention_mask, labels, T_new

    def _prepare_for_generation(self, user_texts: List[str]) -> Tuple[torch.Tensor, torch.Tensor, int]:
        """
        Prepare embeddings and attention mask for generation.

        Args:
            user_texts (List[str]): Input prompts.

        Returns:
            inputs_embeds (torch.Tensor): Embeddings with adversarial block.
            attention_mask (torch.Tensor): Attention mask.
            prefix_length (int): Length of the prefix (to slice off generated tokens).
        """
        assert self.adversarial_embeddings is not None
        B = len(user_texts)
        # Only user role
        convos = [
            [{"role": "user", "content": user + " " + self.tokenizer.convert_ids_to_tokens(self.adv_token_id)}] for user in user_texts
        ]
        toks = self.tokenizer.apply_chat_template(
            convos,
            add_generation_prompt=True,
            continue_final_message=False,
            padding=True,
            padding_side="right",
            return_tensors="pt",
            return_dict=True,
        ).to(self.adversarial_embeddings.device)
        input_ids = toks.input_ids
        attn_mask = toks.attention_mask
        orig_emb = self.model.get_input_embeddings()(input_ids)
        B, T = input_ids.size()
        positions = [(row == self.adv_token_id).nonzero(as_tuple=True)[0].item() for row in input_ids]
        N = self.num_tokens
        D = orig_emb.size(-1)
        T_new = T - 1 + N

        inputs_embeds = torch.zeros(B, T_new, D, device=orig_emb.device)
        new_mask = torch.zeros(B, T_new, device=orig_emb.device)

        for i in range(B):
            pos = positions[i]
            inputs_embeds[i, :pos] = orig_emb[i, :pos]
            new_mask[i, :pos] = attn_mask[i, :pos]
            block = self.adversarial_embeddings if self.universal else self.adversarial_embeddings[i]
            inputs_embeds[i, pos : pos + N] = block
            new_mask[i, pos : pos + N] = 1
            inputs_embeds[i, pos + N :] = orig_emb[i, pos + 1 :]
            new_mask[i, pos + N :] = attn_mask[i, pos + 1 :]

        return inputs_embeds, new_mask, T_new

    def generate(self, user_texts: List[str], max_new_tokens: int = 50, **gen_kwargs: Any) -> List[str]:
        """
        Generate continuations using adversarial embeddings.

        Args:
            user_texts (List[str]): Input prompts.
            max_new_tokens (int): Number of tokens to generate.
            gen_kwargs: Additional generate() keyword args.

        Returns:
            List[str]: Generated text continuations.
        """
        inputs_embeds, attention_mask, prefix_len = self._prepare_for_generation(user_texts)
        bad = gen_kwargs.get("bad_words_ids", [])
        bad.append([self.adv_token_id])
        
        outputs = self.model.generate(
            text_inputs=None,
            inputs_embeds=inputs_embeds, 
            attention_mask=attention_mask, 
            max_new_tokens=max_new_tokens, 
            bad_words_ids=bad, 
            **gen_kwargs
        )
        
        # slice off prefix
        # gen_ids = outputs[:, prefix_len:]
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=True)


class EmbeddingAttacker:
    """
    Base attacker: runs a generic attack loop on a single batch.
    Subclasses implement `_update()`.
    """

    def __init__(self, poisoner: EmbedPoisoner, steps: int = 100) -> None:
        self.poisoner = poisoner
        self.steps = steps

    def fit(self, user_texts: List[str], target_texts: List[str]) -> torch.Tensor:
        """
        Perform the attack, returning the optimized adversarial embeddings.

        Args:
            user_texts (List[str]): Input prompts.
            target_texts (List[str]): Desired targets (for training).

        Returns:
            torch.Tensor: The optimized adversarial embeddings.
        """
        B = len(user_texts)
        self.poisoner.init_embeddings(B)
        for _ in range(self.steps):
            inputs_embeds, attn_mask, labels, _ = self.poisoner._prepare_for_training(user_texts, target_texts)
            loss = self.poisoner.model(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels).loss
            self._update(loss)
            self.poisoner.project()
        return self.poisoner.adversarial_embeddings.detach()

    def _update(self, loss: torch.Tensor) -> None:
        """Attack-specific update rule. Must be overridden."""
        raise NotImplementedError()


class OptimAttack(EmbeddingAttacker):
    """Attack using an optimizer created via an optimizer_factory."""

    def __init__(self, poisoner: EmbedPoisoner, optimizer_factory: Callable[[nn.Parameter], Optimizer], steps: int = 100) -> None:
        super().__init__(poisoner, steps)
        self.optimizer_factory = optimizer_factory
        self.optimizer: Optional[Optimizer] = None

    def fit(self, user_texts: List[str], target_texts: List[str]) -> torch.Tensor:
        B = len(user_texts)
        self.poisoner.init_embeddings(B)
        # create optimizer after adv_emb exists
        self.optimizer = self.optimizer_factory(self.poisoner.adversarial_embeddings)
        for _ in tqdm(range(self.steps), desc="OptimAttack"):  # type: ignore
            self.optimizer.zero_grad()
            inputs_embeds, attn_mask, labels, _ = self.poisoner._prepare_for_training(user_texts, target_texts)
            loss = self.poisoner.model(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels).loss
            loss.backward()
            self.optimizer.step()
            self.poisoner.project()
        return self.poisoner.adversarial_embeddings.detach()

    def _update(self, loss: torch.Tensor) -> None:
        pass  # not used; update logic in fit


class FGSMAttack(EmbeddingAttacker):
    """Fast Gradient Sign Method attacker."""

    def __init__(self, poisoner: EmbedPoisoner, epsilon: float = 0.01, steps: int = 1) -> None:
        super().__init__(poisoner, steps)
        self.epsilon = epsilon

    def _update(self, loss: torch.Tensor) -> None:
        """Perform one FGSM step on adversarial embeddings."""
        self.poisoner.adversarial_embeddings.grad = None
        loss.backward()
        with torch.no_grad():
            self.poisoner.adversarial_embeddings += self.epsilon * self.poisoner.adversarial_embeddings.grad.sign()
            self.poisoner.adversarial_embeddings.grad.zero_()

    def fit(self, user_texts: List[str], target_texts: List[str]) -> torch.Tensor:
        B = len(user_texts)
        self.poisoner.init_embeddings(B)
        for _ in range(self.steps):
            inputs_embeds, attn_mask, labels, _ = self.poisoner._prepare_for_training(user_texts, target_texts)
            loss = self.poisoner.model(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels).loss
            self._update(loss)
            self.poisoner.project()
        return self.poisoner.adversarial_embeddings.detach()

KeyboardInterrupt: 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import optim

model_name = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

pois = EmbedPoisoner(model=model, tokenizer=tokenizer, adv_token="<|adv|>", num_tokens=10, universal=False, norm="inf", epsilon=1.0)
pois.to("cuda")

inputs = ["Translate to French: Hello!", "How are you?", "What is the meaning of life?"]
labels = ["Bonjour !", "Im good, thanks!", "45 actually"]

# Adam Attack
adam = OptimAttack(pois, optimizer_factory=lambda p: optim.Adam([p], lr=1e-4), steps=100)
adv_emb = adam.fit(inputs, labels)
print("Adam =>", pois.generate(inputs, max_new_tokens=20))

# FGSM Attack
fgsm = FGSMAttack(pois, epsilon=0.05, steps=1)
adv_emb2 = fgsm.fit(inputs, labels)
print("FGSM =>", pois.generate(inputs, max_new_tokens=20))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


OptimAttack:   0%|          | 0/100 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Adam => ['assistant\n\nBonjour!  La réponse est : Bonjour!', "assistant\n\nIm good, thanks! Here's why:\n\n- IM (I'm) because", "45 actually \n\n45 actually is the actual number of letters in the word '45'."]


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


FGSM => ['a 1 2 3 4 5 6 7 8 9 ', '', "I can't have a definitive answer to this question.\n\n\x08\n\nThe meaning of life is a complex"]
